# Template Model

## Cara memakai notebook ini

Notebook ini sudah menyiapkan **data** dan **penilaian** untukmu. Tugasmu hanya satu: **mengisi modelnya**.

1. **Salin dulu** file ini, jangan edit aslinya. Beri nama sesuai modelmu, misalnya `01_linear_regression.ipynb`, `02_random_forest.ipynb`, atau `03_xgboost.ipynb`.
2. Cari sel bertanda **`# === ISI BAGIAN INI ===`**. Hanya sel itu yang boleh kamu ubah.
3. Di sel itu, ganti `NAMA_MODEL` dengan nama modelmu, lalu hapus tanda `#` di depan **satu** baris contoh model yang sesuai.
4. Jalankan semua sel dari atas: menu **Run → Run All Cells** (atau *Restart & Run All*).
5. Lihat tabel di bagian paling bawah. Modelmu harus mengalahkan baris `baseline_lag1`.

## Aturan

- **Jangan ubah sel bertanda `# === JANGAN DIUBAH ===`.** Sel itu menjamin ketiga model memakai data dan cara penilaian yang sama. Kalau diubah, hasilmu tidak bisa dibandingkan dengan milik anggota lain.
- **Jangan pernah memakai `X_test` / `y_test` untuk memilih pengaturan model.** Data 2025 hanya untuk penilaian akhir. Untuk mencoba-coba pengaturan, pakai `lipatan_waktu()` (lihat bagian *Opsional* di bawah).
- Setiap model otomatis dilatih **4 kali**: 2 skenario (`dengan_lag`, `tanpa_lag`) × 2 versi target (`asli`, `log`). Kamu tidak perlu menulis perulangannya.
- Hasil tersimpan otomatis di `hasil/evaluasi.csv`. Selama `NAMA_MODEL` belum diganti, hasil **tidak** disimpan.

## Kalau muncul error

Baca baris **terakhir** pesan error. Sebagian besar sudah ditulis dalam Bahasa Indonesia oleh `fondasi.py` dan menjelaskan apa yang salah. Kalau masih bingung, kirim pesan error lengkapnya ke tim.

Aturan lengkap ada di `docs/STANDAR_PENGERJAAN.md`.

In [ ]:
# === JANGAN DIUBAH ===
import sys
from pathlib import Path

# Cari folder akar proyek (yang berisi folder src/), lalu daftarkan src/
AKAR = Path.cwd()
while not (AKAR / "src" / "fondasi.py").exists():
    AKAR = AKAR.parent
sys.path.insert(0, str(AKAR / "src"))

import pandas as pd
from fondasi import (muat_data, evaluasi, info_baris, lipatan_waktu,
                     ke_log, dari_log, SKENARIO, FILE_EVALUASI, RANDOM_STATE)

DATA = {}
for skenario in SKENARIO:
    for jalur in (None, "SNBP", "SNBT"):
        DATA[(skenario, jalur)] = muat_data(skenario, jalur, verbose=False)

pd.DataFrame(
    [(sk, jl or "gabungan", len(d[0]), len(d[1]), d[0].shape[1]) for (sk, jl), d in DATA.items()],
    columns=["skenario", "jalur", "baris_latih", "baris_uji", "jumlah_fitur"])

In [ ]:
# === ISI BAGIAN INI ===

# 1. Nama modelmu (huruf kecil, pakai garis bawah). Contoh: "random_forest"
NAMA_MODEL = "isi_nama_model"

# 2. Jalur yang dipakai. None = gabungan SNBP+SNBT (jalur jadi fitur).
#    Boleh ditambah "SNBP" dan/atau "SNBT" untuk melatih per jalur.
JALUR_DIPAKAI = [None]

# 3. Model. Hapus tanda # di depan SATU baris "return ..." yang sesuai,
#    lalu hapus baris "return DummyRegressor" paling bawah.
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

def buat_model():
    # return make_pipeline(StandardScaler(), LinearRegression())
    # return RandomForestRegressor(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1)
    # return XGBRegressor(n_estimators=500, learning_rate=0.05, max_depth=6, random_state=RANDOM_STATE)
    return DummyRegressor(strategy="mean")   # pengganti sementara: selalu menebak rata-rata

### Opsional · Menyetel pengaturan model (hyperparameter)

Kalau ingin mencari pengaturan terbaik, **selalu** pakai `cv=lipatan_waktu(X_train)`, jangan `cv=5`. `cv=5` mengacak data sehingga model belajar dari masa depan dan skornya terlalu bagus. Contoh (salin ke sel baru di atas sel *JANGAN DIUBAH* berikutnya):

```python
from sklearn.model_selection import GridSearchCV
X_train, X_test, y_train, y_test = DATA[("dengan_lag", None)]
cari = GridSearchCV(
    RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1),
    param_grid={"n_estimators": [200, 400], "max_depth": [None, 20]},
    cv=lipatan_waktu(X_train), scoring="neg_mean_absolute_error")
cari.fit(X_train, y_train)
print(cari.best_params_)   # lalu pakai angka ini di buat_model()
```

In [ ]:
# === JANGAN DIUBAH ===
SIMPAN = NAMA_MODEL != "isi_nama_model"
if not SIMPAN:
    print("NAMA_MODEL belum diganti -> hasil hanya ditampilkan, TIDAK disimpan.\n")

for skenario in SKENARIO:
    for jalur in JALUR_DIPAKAI:
        X_train, X_test, y_train, y_test = DATA[(skenario, jalur)]
        for target in ("asli", "log"):
            model = buat_model()
            if target == "asli":
                model.fit(X_train, y_train)
                prediksi = model.predict(X_test)
            else:
                model.fit(X_train, ke_log(y_train))
                prediksi = dari_log(model.predict(X_test))
            evaluasi(y_test, prediksi, NAMA_MODEL, skenario, jalur, target=target,
                     n_latih=len(X_train), n_fitur=X_train.shape[1], simpan=SIMPAN)
            print()

In [ ]:
# === JANGAN DIUBAH ===
# Hasil modelmu dibandingkan dengan baseline
if FILE_EVALUASI.exists():
    hasil = pd.read_csv(FILE_EVALUASI)
    tampil = hasil[hasil["nama_model"].isin([NAMA_MODEL, "baseline_lag1", "baseline_rata_kelompok"])
                   & hasil["jalur"].isin([j or "gabungan" for j in JALUR_DIPAKAI])]
    tampil = tampil[["nama_model", "skenario", "jalur", "target", "MAE", "MAPE", "R2", "RMSLE"]]
    display(tampil.sort_values(["skenario", "jalur", "MAE"]).reset_index(drop=True))
else:
    print("hasil/evaluasi.csv belum ada. Jalankan notebooks/00_baseline.ipynb terlebih dahulu.")